# MBH-TOPTW — Campaña computacional (Gurobi WLS en Colab)

Reproduce **todas** las tablas del paper: Tabla 2 (benchmark Cordeau adaptado, heterogéneo), Tabla 3 (escalabilidad),
Tabla 5 (instancias aleatorias, 20 escenarios × 5 réplicas) y la instancia real de la Sección 6.4 (14 sectores de 8 AMP, 5 bases, 5 aeronaves de 3 clases).

**Antes de correr** (una sola vez):
1. Licencia **Gurobi Academic WLS** (gratis): se obtiene gratis en https://www.gurobi.com/academia/academic-program-and-licenses/. Obtienes tres valores:
   `WLSACCESSID`, `WLSSECRET`, `LICENSEID`.
2. En Colab, panel izquierdo → **🔑 Secrets** → agrega esos tres nombres con sus valores y activa "Notebook access".
   (Alternativa: sube el archivo `gurobi.lic` a `MyDrive/mbh_toptw/gurobi.lic`.)
3. Sube `mbh_toptw_package.zip` a `MyDrive/mbh_toptw/`.

**Cómo se corre:** celdas en orden. Cada corrida terminada se guarda al instante en `MyDrive/mbh_toptw/results/runs.csv`;
si la sesión se corta, vuelves a ejecutar y **continúa donde quedó** (salta lo ya hecho).

**Versión de modelo:** cada fila de `runs.csv` lleva la columna `model`. Las filas escritas por una versión anterior del
código (o con error) se conservan en el archivo pero se ignoran en las tablas y la corrida se repite. El paquete (zip) se
descomprime en el disco local de Colab; reemplazarlo en Drive **no toca** la carpeta `results/`.

| campaña | corridas | límite por corrida | estimación |
|---|---|---|---|
| benchmark (T2) | 20 inst × {MILP, CG, CG-TS} | MILP 3600 s, CG 3600 s, CG-TS libre | ~20–25 h |
| escalabilidad (T3) | pr02: 8 inst × 3; pr06: 7 inst × 3 | MILP 3600 s, CG 3600 s | pr02 < 5 min; pr06 ~6–10 h |
| aleatorias (T5) | 100 inst × 3 | MILP 1800 s, CG 1800 s | ~40–70 h |
| real (Sec. 6.4) | 1 instancia × 3 | 3600 s | < 5 min |

La celda 4 define `SESSION_BUDGET_H`: la campaña deja de lanzar corridas nuevas al agotarse ese presupuesto,
para que todo quede guardado antes de que Colab corte la sesión (12 h en la versión gratuita).

## 1. Dependencias y licencia

In [ ]:
!pip -q install "gurobipy>=12" pandas pyarrow scipy
import os, sys, json, time, shutil, glob
import numpy as np, pandas as pd

# --- licencia WLS: desde Colab Secrets (preferido) o desde Drive ---
LIC_PATH = '/content/gurobi.lic'
try:
    from google.colab import userdata
    acc, sec, lid = userdata.get('WLSACCESSID'), userdata.get('WLSSECRET'), userdata.get('LICENSEID')
    with open(LIC_PATH, 'w') as f:
        f.write(f"WLSACCESSID={acc}\nWLSSECRET={sec}\nLICENSEID={lid}\n")
    print('licencia escrita desde Colab Secrets')
except Exception as e:
    print('Secrets no disponibles (', str(e)[:60], ') -> se buscara MyDrive/mbh_toptw/gurobi.lic tras montar Drive')
os.environ['GRB_LICENSE_FILE'] = LIC_PATH

## 2. Drive, paquete y resultados

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/mbh_toptw'
os.makedirs(BASE + '/results', exist_ok=True)
if not os.path.exists(LIC_PATH) or os.path.getsize(LIC_PATH) < 20:
    assert os.path.exists(BASE + '/gurobi.lic'), 'No hay licencia: usa Colab Secrets o sube gurobi.lic a MyDrive/mbh_toptw/'
    shutil.copy(BASE + '/gurobi.lic', LIC_PATH)

# paquete (codigo + instancias)
PKG = BASE + '/mbh_toptw_package.zip'
assert os.path.exists(PKG), 'Sube mbh_toptw_package.zip a MyDrive/mbh_toptw/'
shutil.rmtree('/content/mbh', ignore_errors=True); shutil.unpack_archive(PKG, '/content/mbh')
CODE = glob.glob('/content/mbh/**/mbh_core.py', recursive=True)[0].rsplit('/', 1)[0]
INST = CODE.rsplit('/', 1)[0] + '/instances'
sys.path.insert(0, CODE)
from mbh_core import *
from mbh_experiments import *
import mbh_instances
print('codigo:', CODE); print('instancias:', {d: len(os.listdir(f'{INST}/{d}')) for d in ['benchmark_het', 'scalability', 'scalability_hard', 'random'] if os.path.isdir(f'{INST}/{d}')})

# --- Gurobi: un solo entorno (sesion WLS) para toda la campana ---
import gurobipy as gp
from gurobipy import GRB
ENV = gp.Env(); ENV.setParam('OutputFlag', 0)
m = gp.Model(env=ENV); x = m.addVars(5000, ub=1.0); m.setObjective(x.sum(), GRB.MAXIMIZE); m.optimize()
print('Gurobi', gp.gurobi.version(), '| test LP de 5000 variables:', m.Status == GRB.OPTIMAL, '(si falla aqui la licencia es la limitada de pip)')
m.dispose()
HOST = f"colab-{os.cpu_count()}cpu"
RES = Campaign(BASE + '/results', env=ENV, threads=0, host=HOST)
print(f'version de modelo del codigo: {MODEL_VERSION} | corridas validas ya guardadas: {len(RES.clean())} | filas antiguas o con error (se ignoran y se repiten): {len(RES.stale())}')

## 3. Configuración

Los límites se aplican por corrida. `SESSION_BUDGET_H` corta el lanzamiento de corridas nuevas cuando la sesión lleva
ese tiempo (las corridas en curso terminan). Marca `RUN_*` para elegir qué campañas lanzar en esta sesión.

In [ ]:
T_SESSION0 = time.time()
SESSION_BUDGET_H = 10.5          # horas de esta sesion (Colab gratis corta a ~12 h)
TL_MILP_BENCH, TL_CG_BENCH, SP_TIME = 3600.0, 3600.0, 60.0
TL_MILP_SCAL,  TL_CG_SCAL          = 3600.0, 3600.0
TL_MILP_RAND,  TL_CG_RAND          = 1800.0, 1800.0
TL_REAL                            = 3600.0
FINAL_MIP_TIME = 600.0            # MIP final sobre columnas (CG y CG-TS)

RUN_BENCH, RUN_SCAL, RUN_RANDOM, RUN_REAL = True, True, True, True
SCENARIO_FILTER = None            # p.ej. ['rnd01','rnd02'] para una sesion parcial; None = todos
METHODS = ['CG-TS', 'CG', 'MILP'] # orden: primero lo rapido, para tener resultados parciales utiles

def budget_left():
    return (time.time() - T_SESSION0) / 3600 < SESSION_BUDGET_H
def launch(campaign, path, method, tl, **kw):
    if not budget_left():
        return 'budget'
    row = RES.run(campaign, path, method, tl, sp_time=SP_TIME, final_mip_time=FINAL_MIP_TIME, **kw)
    if row is None: return 'skip'
    print(f"  {campaign:11s} {os.path.basename(path):26s} {method:6s} value={row.get('value')} bound={row.get('bound')} "
          f"gap={None if row.get('gap') is None else round(row['gap'],2)} time={None if row.get('time') is None else round(row['time'],1)} {row.get('status')}", flush=True)
    return 'ok'

## 4. Autotest (≈2–5 min): pr11, pr01 y la instancia real con los tres métodos

Verifica factibilidad de todas las rutas (evaluador independiente), validez de las cotas y, en la instancia real, que el
MILP alcance el óptimo exacto conocido por enumeración completa (188), que requiere rutas que vuelven a su base de salida.

In [ ]:
KNOWN_OPT = {'real_chile': 188.0}     # optimo por enumeracion completa (brute_force_optimum), incluye rutas cerradas
os.makedirs(f'{INST}/real', exist_ok=True)
write_instance(mbh_instances.build_real_instance(name='real_chile'), f'{INST}/real/real_chile')
for path in [f'{INST}/benchmark_het/pr11', f'{INST}/benchmark_het/pr01', f'{INST}/real/real_chile']:
    nm = os.path.basename(path); inst = read_instance(path, nm)
    r_m = solve_milp(inst, time_limit=300, env=ENV)
    r_c = run_cg(inst, 'CG', time_limit=300, env=ENV, sp_time=30, verbose=False)
    r_t = run_cg(inst, 'CG-TS', time_limit=300, env=ENV, verbose=False)
    okm, vm = verify_solution(inst, r_m['routes']); okc, vc = verify_solution(inst, r_c.routes); okt, vt = verify_solution(inst, r_t.routes)
    n_closed = sum(1 for rt in r_m['routes'] if rt['route'] and rt['start'] == rt['end'])
    print(f"{nm}: MILP={r_m['value']} ({r_m['status']}, {r_m['time']:.1f}s, bound {r_m['bound']:.2f}, rutas cerradas {n_closed}) | "
          f"CG int={r_c.integer_value} UB={r_c.lagr_bound:.2f} proved={r_c.proved} ({r_c.time_total:.1f}s) | CG-TS int={r_t.integer_value} ({r_t.time_total:.1f}s)")
    assert okm and okc and okt, 'solucion infactible'
    assert abs(vm - r_m['value']) < 1e-6 and abs(vc - r_c.integer_value) < 1e-6 and abs(vt - r_t.integer_value) < 1e-6, 'valor reportado != valor verificado'
    assert r_c.lagr_bound <= inst.score.sum() + 1e-6, f'cota CG {r_c.lagr_bound:.1f} > suma de scores {inst.score.sum():.0f}: relajacion invalida'
    assert r_m['bound'] <= inst.score.sum() + 1e-6, 'cota MILP > suma de scores'
    if r_m['status'] == 'optimal':
        assert r_c.lagr_bound >= r_m['value'] - 1e-6, 'la cota de CG no domina al optimo del MILP'
        assert max(r_c.integer_value, r_t.integer_value) <= r_m['value'] + 1e-6, 'una heuristica supera al optimo exacto: revisar'
        if nm in KNOWN_OPT: assert abs(r_m['value'] - KNOWN_OPT[nm]) < 1e-6, f'MILP={r_m["value"]} != optimo conocido {KNOWN_OPT[nm]}'
print('autotest OK')

## 5. Campaña benchmark (Tabla 2): 20 instancias heterogéneas × {CG-TS, CG, MILP}

In [ ]:
if RUN_BENCH:
    order = [f'pr{k:02d}' for k in (11, 17, 1, 7, 12, 2, 18, 8, 13, 3, 19, 9, 14, 4, 20, 10, 15, 5, 16, 6)]   # chicas primero
    for nm in order:
        for mth in METHODS:
            st = launch('benchmark', f'{INST}/benchmark_het/{nm}', mth, TL_MILP_BENCH if mth == 'MILP' else TL_CG_BENCH)
            if st == 'budget': print('presupuesto de sesion agotado'); break
        if not budget_left(): break
    print('benchmark: corridas guardadas =', len(RES.clean()[RES.clean().campaign == 'benchmark']))

## 6. Escalabilidad (Tabla 3)

Dos familias con la flota fija y el número de áreas creciente: **pr02** (3 aviones, 20…90 áreas, la de la Tabla 3
original) y **pr06** (7 aviones, 40…280 áreas), donde el MILP deja de cerrar. Primero se corre pr02 (ya hecha: se salta).

In [ ]:
SCAL_FAMILIES = [('scalability', 'scalability'), ('scalability_hard', 'scalability_hard')]   # (campana, carpeta)
if RUN_SCAL:
    for camp, folder in SCAL_FAMILIES:
        if not os.path.isdir(f'{INST}/{folder}'): print('falta la carpeta', folder, '(zip antiguo)'); continue
        for nm in sorted(os.listdir(f'{INST}/{folder}')):
            for mth in METHODS:
                st = launch(camp, f'{INST}/{folder}/{nm}', mth, TL_MILP_SCAL if mth == 'MILP' else TL_CG_SCAL)
                if st == 'budget': break
            if not budget_left(): break
        if not budget_left(): print('presupuesto de sesion agotado'); break
    d = RES.clean(); print('escalabilidad: corridas guardadas =', {c: int((d.campaign == c).sum()) for c, _ in SCAL_FAMILIES})

## 7. Instancias aleatorias (Tabla 5): 20 escenarios × 5 réplicas × 3 métodos

In [ ]:
if RUN_RANDOM:
    names = sorted(os.listdir(f'{INST}/random'))
    if SCENARIO_FILTER: names = [n for n in names if n.split('_')[0] in SCENARIO_FILTER]
    # orden por tamano (areas) para acumular resultados completos por escenario
    names.sort(key=lambda n: (int(n.split('_')[1].split('a')[1].split('v')[0]), n))
    for nm in names:
        for mth in METHODS:
            st = launch('random', f'{INST}/random/{nm}', mth, TL_MILP_RAND if mth == 'MILP' else TL_CG_RAND)
            if st == 'budget': break
        if not budget_left(): print('presupuesto de sesion agotado'); break
    d = RES.clean(); print('random: corridas guardadas =', int((d.campaign == 'random').sum()), 'de', 3 * len(names))

## 8. Instancia real (Sección 6.4): un día operacional sobre las AMP de Chile

14 sectores de patrulla derivados de las 8 AMP de la hoja de Darío (las oceánicas y la austral divididas en sectores),
5 bases (las 4 de la hoja + Mataveri como base avanzada), 5 aeronaves de 3 clases (P-3ACH, C-295, Dauphin) con
límite de ruta de 8 h por jornada de tripulación, Koopman con alcance lateral $\rho$, $P_{\min}$ por sector,
ventanas de luz diurna del 21 de junio calculadas por latitud. Todo documentado en `docs/MEMO_instancia_real.md`.
Se puede cambiar el día (`DOY`), la importancia por sector y la flota; ver `mbh_instances.REAL_*`.

In [ ]:
DOY = 172                      # dia del anio de planificacion (172 = 21 de junio, dia mas corto)
IMPORTANCE_OVERRIDE = {}       # p.ej. {'Paso Drake E': 5}
if RUN_REAL:
    os.makedirs(f'{INST}/real', exist_ok=True); os.makedirs(BASE + '/results/tables', exist_ok=True)
    inst = mbh_instances.build_real_instance(doy=DOY, importance=IMPORTANCE_OVERRIDE or None, name='real_chile')
    write_instance(inst, f'{INST}/real/real_chile'); json.dump(inst.meta, open(f'{INST}/real/real_chile_meta.json', 'w'), indent=1)
    T, F = mbh_instances.real_instance_tables(inst, BASE + '/results/tables')
    print(T.to_string(index=False)); print(); print(F.to_string(index=False))
    for mth in METHODS:
        launch('real', f'{INST}/real/real_chile', mth, TL_REAL, force=True)
    shutil.copytree(f'{INST}/real', BASE + '/results/real_instance', dirs_exist_ok=True)
    names = [a['sector'] for a in inst.meta['areas']]; bnames = [b['name'] for b in inst.meta['bases']]; acl = [a['cls'] for a in inst.meta['aircraft']]
    for mth in METHODS:
        f = f"{BASE}/results/routes/real__real_chile__{mth}.json"
        if os.path.exists(f):
            for rt in json.load(open(f)):
                if isinstance(rt, dict) and 'route' in rt:
                    print(f"  {mth:6s} {acl[rt['r']]:16s}: {bnames[rt['start']]} -> " + ' -> '.join(names[k] for k in rt['route']) + f" -> {bnames[rt['end']]}  (score {rt['score']:.0f}, {rt['dur']:.0f} min)")

## 9. Tablas, tests estadísticos, resumen y descarga (se puede ejecutar en cualquier momento, con lo que haya)

In [ ]:
OUT = BASE + '/results/tables'; os.makedirs(OUT, exist_ok=True)
d = RES.clean()
S = []; A = S.append
A('=' * 78); A(f'MBH-TOPTW  resumen  {time.strftime("%Y-%m-%d %H:%M")}  host={HOST}  gurobi={gp.gurobi.version()}'); A('=' * 78)
A(f'corridas sin error: {len(d)}   por campana: {d.campaign.value_counts().to_dict()}')

# ---- T2 benchmark ----
bt = benchmark_table(d, 'benchmark')
if len(bt):
    cols = ['instance', 'milp_value', 'milp_time', 'milp_gap', 'cg_value', 'cg_time', 'cg_ub', 'cg_gap', 'cg_proved', 'ts_value', 'ts_time', 'ts_gap']
    bt2 = bt.reindex(columns=cols)
    to_latex(bt2, f'{OUT}/T2_benchmark.tex', 'Results on the heterogeneous adapted-Cordeau instances. MILP gap: solver gap at termination; '
             'CG UB: Lagrangian/LP upper bound of the set-packing reformulation; CG and CG-TS gaps are relative to the best upper bound (min of MILP bound and CG UB).',
             'tab2', fmt={'milp_value': '{:.0f}', 'milp_time': '{:.1f}', 'milp_gap': '{:.2f}', 'cg_value': '{:.0f}', 'cg_time': '{:.1f}', 'cg_ub': '{:.2f}',
                          'cg_gap': '{:.2f}', 'ts_value': '{:.0f}', 'ts_time': '{:.1f}', 'ts_gap': '{:.2f}'})
    A(''); A('--- T2 benchmark ---'); A(bt2.round(2).to_string(index=False))
    sb = paired_stats(d[d.campaign == 'benchmark'], 'time'); sv = paired_stats(d[d.campaign == 'benchmark'], 'value')
    A(f'tests (tiempo): {sb}'); A(f'tests (valor): {sv}')
    A(f"CG >= MILP value in {int((bt.cg_value >= bt.milp_value - 1e-9).sum())}/{int(bt.milp_value.notna().sum())} instances; "
      f"CG-TS mean gap to best UB = {bt.ts_gap.mean():.2f}% (max {bt.ts_gap.max():.2f}%); CG mean gap = {bt.cg_gap.mean():.2f}%")

# ---- T3 scalability (two families) ----
for camp, tex, lab, desc in [('scalability', 'T3_scalability', 'tab3', 'pr02 family, 3 aircraft'),
                             ('scalability_hard', 'T3b_scalability_pr06', 'tab3b', 'pr06 family, 7 aircraft')]:
    sc = d[d.campaign == camp]
    if not len(sc): continue
    P = sc.pivot_table(index='instance', columns='method', values='time', aggfunc='last')
    P['areas'] = [int(i.split('_n')[1]) for i in P.index]; P = P.sort_values('areas').reset_index()
    V = sc.pivot_table(index='instance', columns='method', values='value', aggfunc='last')
    P = P.merge(V.add_suffix('_value').reset_index(), on='instance')
    G = sc[sc.method == 'MILP'].set_index('instance').gap.rename('MILP_gap')
    U = sc[sc.method == 'CG'].set_index('instance').bound.rename('CG_ub')
    P = P.merge(G, left_on='instance', right_index=True, how='left').merge(U, left_on='instance', right_index=True, how='left')
    cols = ['areas'] + [c for c in ['MILP', 'CG', 'CG-TS', 'MILP_value', 'MILP_gap', 'CG_value', 'CG_ub', 'CG-TS_value'] if c in P]
    to_latex(P[cols], f'{OUT}/{tex}.tex',
             f'Processing time (s), objective value, MILP gap (\\%) and CG upper bound as the number of areas grows ({desc}, heterogeneous fleet).', lab,
             fmt={'MILP': '{:.1f}', 'CG': '{:.1f}', 'CG-TS': '{:.1f}', 'MILP_value': '{:.0f}', 'MILP_gap': '{:.2f}', 'CG_value': '{:.0f}',
                  'CG_ub': '{:.1f}', 'CG-TS_value': '{:.0f}'}, wide=False)
    A(''); A(f'--- {tex} ({desc}) ---'); A(P[cols].round(2).to_string(index=False))
    if 'MILP' in P and 'CG' in P and len(P) >= 5:
        A(f"Wilcoxon CG faster than MILP: p={wilcoxon_paired(P['MILP'], P['CG'], 'greater')[1]:.4f}; speedup median={np.median(P['MILP']/np.maximum(P['CG'],1e-3)):.1f}x")
        lg = np.log(P['areas'])
        for mth in ['MILP', 'CG', 'CG-TS']:
            if mth in P and (P[mth] > 0).all(): A(f"  growth exponent {mth}: {np.polyfit(lg, np.log(P[mth]), 1)[0]:.2f}")

# ---- T5 random ----
rt, dr = random_table(d, 'random')
if len(rt):
    cols = ['scenario', 'R', 'V', 'D', 'n', 'milp_time', 'milp_sd', 'milp_gap', 'cg_time', 'cg_sd', 'cg_gap', 'ts_time', 'ts_sd', 'ts_gap']
    rt2 = rt.reindex(columns=cols)
    to_latex(rt2, f'{OUT}/T5_random.tex', 'Randomised scenarios: mean and standard deviation of solution time (s) over replications, and mean gap (\\%) to the best upper bound.',
             'tab5', fmt={c: '{:.1f}' for c in cols[5:]})
    A(''); A('--- T5 random ---'); A(rt2.round(2).to_string(index=False))
    A(f"tests (tiempo): {paired_stats(dr, 'time')}"); A(f"tests (valor): {paired_stats(dr, 'value')}")
    g = dr.groupby('method').agg(mean_gap=('gap_ub', 'mean'), max_gap=('gap_ub', 'max'), mean_time=('time', 'mean'), n=('instance', 'nunique'))
    A(g.round(2).to_string())

# ---- real ----
rr = d[d.campaign == 'real'].drop_duplicates(['instance', 'method'], keep='last')
if len(rr):
    cols = ['instance', 'method', 'R', 'V', 'D', 'value', 'bound', 'gap', 'time', 'status']
    to_latex(rr[cols], f'{OUT}/T6_real.tex', 'Real MPA surveillance instance (14 sectors, 5 bases, 5 aircraft of 3 classes): results by method.', 'tab6',
             fmt={'value': '{:.0f}', 'bound': '{:.1f}', 'gap': '{:.2f}', 'time': '{:.1f}'})
    A(''); A('--- real ---'); A(rr[cols].round(2).to_string(index=False))

A('=' * 78); txt = '\n'.join(S); print(txt); open(f'{OUT}/SUMMARY.txt', 'w').write(txt)
d.to_csv(f'{OUT}/runs_clean.csv', index=False)
z = shutil.make_archive(BASE + '/results_mbh', 'zip', BASE + '/results')
from google.colab import files
files.download(z); files.download(f'{OUT}/SUMMARY.txt')